# UFC Stats Scraper\n\nScrapes event, fight, and fighter data from ufcstats.com using 10 concurrent threads.\n\n## Important Discovery\n- The **event page** always lists the **winner first** (not the red corner)\n- The **fight detail page** lists the **red corner first** with W/L status\n- We scrape both, then rebuild fights.csv using the detail page for correct corner assignment\n- **Fighter 1 = Red corner = Favorite**, **Fighter 2 = Blue corner = Underdog**

In [ ]:
# Cell 1 — Install Dependencies
!pip install requests beautifulsoup4 pandas numpy lxml tqdm

In [ ]:
# Cell 2 — Imports & Setup
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from tqdm import tqdm
import os
import json
import concurrent.futures
import threading

BASE_URL = "http://www.ufcstats.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"
}
WORKERS = 10
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    response = session.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")

print(f"✅ Ready — using {WORKERS} threads")

In [ ]:
# Cell 3 — Scrape All Events
# FIXED: Date is inside TD[0] alongside event name. TD[1] is location.
def scrape_all_events():
    url = f"{BASE_URL}/statistics/events/completed?page=all"
    soup = get_soup(url)

    events = []
    rows = soup.select("tr.b-statistics__table-row")

    for row in rows:
        link = row.select_one("a.b-link")
        if link:
            cells = row.select("td")

            # Event name and date are both inside TD[0]
            td0_text = cells[0].get_text(separator="|", strip=True)
            parts = td0_text.split("|")

            event_name = parts[0].strip()
            event_date = parts[1].strip() if len(parts) > 1 else ""

            # Location is in TD[1]
            location = cells[1].text.strip() if len(cells) > 1 else ""

            event_url = link["href"]

            events.append({
                "event_name": event_name,
                "event_url": event_url,
                "event_date": event_date,
                "location": location
            })

    print(f"Found {len(events)} events")
    return events

events = scrape_all_events()
events_df = pd.DataFrame(events)
events_df.to_csv(f"{DATA_DIR}/events.csv", index=False)
print(f"✅ Saved {len(events_df)} events")
print(f"\nSample dates:")
print(events_df[["event_name", "event_date", "location"]].head(5).to_string())

In [ ]:
# Cell 4 — Scrape Fights From All Events (10 Workers)
# NOTE: Event page always lists WINNER first. We fix this in Cell 8 using detail pages.
def scrape_event_fights(event_row):
    event_url = event_row["event_url"]
    event_name = event_row["event_name"]
    event_date = event_row["event_date"]

    try:
        soup = get_soup(event_url)
    except Exception as e:
        return []

    fights = []
    rows = soup.select("tr.b-fight-details__table-row")[1:]

    for row in rows:
        cols = row.select("td")
        if len(cols) < 10:
            continue

        fight_url_tag = row.select_one("a.b-flag")
        fight_url = fight_url_tag["href"] if fight_url_tag else None

        fighter_links = cols[1].select("a")
        if len(fighter_links) < 2:
            continue

        fighter1 = fighter_links[0].text.strip()
        fighter2 = fighter_links[1].text.strip()

        def get_col_values(col):
            paragraphs = col.select("p")
            return [p.text.strip() for p in paragraphs]

        kd_vals = get_col_values(cols[2])
        str_vals = get_col_values(cols[3])
        td_vals = get_col_values(cols[4])
        sub_vals = get_col_values(cols[5])

        weight_class = cols[6].text.strip()
        method = cols[7].text.strip()
        fight_round = cols[8].text.strip()
        fight_time = cols[9].text.strip()

        fight_data = {
            "event_name": event_name,
            "event_date": event_date,
            "fight_url": fight_url,
            "fighter_1_winner": fighter1,
            "fighter_2_loser": fighter2,
            "winner_kd": kd_vals[0] if len(kd_vals) > 0 else None,
            "loser_kd": kd_vals[1] if len(kd_vals) > 1 else None,
            "winner_str": str_vals[0] if len(str_vals) > 0 else None,
            "loser_str": str_vals[1] if len(str_vals) > 1 else None,
            "winner_td": td_vals[0] if len(td_vals) > 0 else None,
            "loser_td": td_vals[1] if len(td_vals) > 1 else None,
            "winner_sub": sub_vals[0] if len(sub_vals) > 0 else None,
            "loser_sub": sub_vals[1] if len(sub_vals) > 1 else None,
            "weight_class": weight_class,
            "method": method,
            "round": fight_round,
            "time": fight_time,
        }
        fights.append(fight_data)

    return fights

event_rows = events_df.to_dict("records")
all_fights = []

print(f"Scraping fights from {len(event_rows)} events with {WORKERS} threads...")
print("NOTE: Event page lists winner first. We fix corner order in Cell 8.")

with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(scrape_event_fights, row): row for row in event_rows}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Events"):
        result = future.result()
        if result:
            all_fights.extend(result)

fights_raw_df = pd.DataFrame(all_fights)
fights_raw_df.to_csv(f"{DATA_DIR}/fights_raw.csv", index=False)
print(f"\n✅ Saved {len(fights_raw_df)} raw fights (winner-first order)")
fights_raw_df.head(5)

In [ ]:
# Cell 5 — Scrape Fighter Directory (10 Workers)
def scrape_fighters_for_letter(char):
    url = f"{BASE_URL}/statistics/fighters?char={char}&page=all"
    try:
        soup = get_soup(url)
    except Exception as e:
        return []

    fighters_list = []
    rows = soup.select("tr.b-statistics__table-row")

    for row in rows:
        cols = row.select("td")
        if len(cols) < 10:
            continue

        link = cols[0].select_one("a")
        if not link:
            continue

        fighter = {
            "fighter_url": link["href"],
            "first_name": cols[0].text.strip(),
            "last_name": cols[1].text.strip(),
            "nickname": cols[2].text.strip(),
            "height": cols[3].text.strip(),
            "weight": cols[4].text.strip(),
            "reach": cols[5].text.strip(),
            "stance": cols[6].text.strip(),
            "wins": cols[7].text.strip(),
            "losses": cols[8].text.strip(),
            "draws": cols[9].text.strip(),
        }
        fighters_list.append(fighter)

    return fighters_list

letters = list("abcdefghijklmnopqrstuvwxyz")
all_fighters = []

print(f"Scraping fighter directory with {WORKERS} threads...")

with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(scrape_fighters_for_letter, c): c for c in letters}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Fighter pages"):
        result = future.result()
        if result:
            all_fighters.extend(result)

fighters_df = pd.DataFrame(all_fighters)
fighters_df.to_csv(f"{DATA_DIR}/fighters.csv", index=False)
print(f"\n✅ Saved {len(fighters_df)} fighters")
fighters_df.head(5)

In [ ]:
# Cell 6 — Scrape Detailed Fight Stats (10 Workers)
# This gives us the TRUE red/blue corner order + W/L status
def parse_fight_table(table):
    headers = [th.text.strip() for th in table.select("thead th")]
    rows_data = []
    for row in table.select("tbody tr"):
        cols = row.select("td")
        row_values = []
        for col in cols:
            paragraphs = col.select("p")
            if paragraphs:
                vals = [p.text.strip() for p in paragraphs]
                row_values.append(vals)
            else:
                row_values.append([col.text.strip()])
        rows_data.append(row_values)
    return {"headers": headers, "rows": rows_data}

def scrape_fight_details(fight_url):
    if not fight_url:
        return None
    soup = get_soup(fight_url)
    details = {}

    result_section = soup.select_one("div.b-fight-details__persons")
    if result_section:
        fighters = result_section.select("div.b-fight-details__person")
        for i, fighter in enumerate(fighters, 1):
            name = fighter.select_one("a.b-fight-details__person-link")
            status = fighter.select_one("i.b-fight-details__person-status")
            details[f"fighter_{i}_name"] = name.text.strip() if name else ""
            details[f"fighter_{i}_status"] = status.text.strip() if status else ""

    info_section = soup.select("div.b-fight-details__content p.b-fight-details__text")
    for p in info_section:
        text = p.get_text(separator="|", strip=True)
        items = text.split("|")
        for item in items:
            if ":" in item:
                key, val = item.split(":", 1)
                details[key.strip().lower().replace(" ", "_")] = val.strip()

    tables = soup.select("table.b-fight-details__table")
    if len(tables) >= 1:
        details["totals"] = parse_fight_table(tables[0])
    if len(tables) >= 2:
        details["significant_strikes"] = parse_fight_table(tables[1])

    details["fight_url"] = fight_url
    return details

def scrape_one(url):
    try:
        return scrape_fight_details(url)
    except Exception as e:
        return None

fight_urls = fights_raw_df["fight_url"].dropna().unique()
fight_details_list = []
CHECKPOINT_EVERY = 500

print(f"Scraping {len(fight_urls)} fight detail pages with {WORKERS} threads...")
print("This gives us correct red/blue corner assignment.")

with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(scrape_one, url): url for url in fight_urls}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Fight details"):
        result = future.result()
        if result:
            fight_details_list.append(result)

        if len(fight_details_list) % CHECKPOINT_EVERY == 0 and len(fight_details_list) > 0:
            with open(f"{DATA_DIR}/fight_details_checkpoint.json", "w") as f:
                json.dump(fight_details_list, f)

with open(f"{DATA_DIR}/fight_details.json", "w") as f:
    json.dump(fight_details_list, f)

print(f"\n✅ Saved {len(fight_details_list)} detailed fight records")

In [ ]:
# Cell 7 — Rebuild fights.csv with Correct Red/Blue Corner Order
# Event page: winner always first. Detail page: red corner always first.
# We use the detail page to determine true corner assignment,
# then assign the correct stats from the event page.

detail_lookup = {}
for fd in fight_details_list:
    url = fd.get("fight_url", "")
    f1 = fd.get("fighter_1_name", "").strip()
    f2 = fd.get("fighter_2_name", "").strip()
    s1 = fd.get("fighter_1_status", "").strip()
    s2 = fd.get("fighter_2_status", "").strip()
    if url and f1:
        detail_lookup[url] = {
            "red_corner": f1,
            "blue_corner": f2,
            "red_status": s1,
            "blue_status": s2
        }

rebuilt = []
skipped = 0

for _, row in fights_raw_df.iterrows():
    url = row["fight_url"]

    if url not in detail_lookup:
        skipped += 1
        continue

    d = detail_lookup[url]
    red = d["red_corner"]
    blue = d["blue_corner"]

    # Determine winner from W/L status
    if d["red_status"] == "W":
        winner = red
    elif d["blue_status"] == "W":
        winner = blue
    else:
        winner = "Draw/NC"

    # Event page winner is always fighter_1_winner
    event_winner = row["fighter_1_winner"]

    # Assign stats to correct corner
    if red == event_winner:
        # Red corner won — event stats order: winner=red, loser=blue
        red_kd, blue_kd = row["winner_kd"], row["loser_kd"]
        red_str, blue_str = row["winner_str"], row["loser_str"]
        red_td, blue_td = row["winner_td"], row["loser_td"]
        red_sub, blue_sub = row["winner_sub"], row["loser_sub"]
    else:
        # Blue corner won — event stats order: winner=blue, loser=red
        red_kd, blue_kd = row["loser_kd"], row["winner_kd"]
        red_str, blue_str = row["loser_str"], row["winner_str"]
        red_td, blue_td = row["loser_td"], row["winner_td"]
        red_sub, blue_sub = row["loser_sub"], row["winner_sub"]

    rebuilt.append({
        "event_name": row["event_name"],
        "event_date": row["event_date"],
        "fight_url": url,
        "fighter_1": red,       # Red corner (favorite)
        "fighter_2": blue,      # Blue corner (underdog)
        "winner": winner,
        "f1_kd": red_kd,
        "f2_kd": blue_kd,
        "f1_str": red_str,
        "f2_str": blue_str,
        "f1_td": red_td,
        "f2_td": blue_td,
        "f1_sub": red_sub,
        "f2_sub": blue_sub,
        "weight_class": row["weight_class"],
        "method": row["method"],
        "round": row["round"],
        "time": row["time"],
    })

fights_df = pd.DataFrame(rebuilt)

# Verify
f1_win = (fights_df["winner"] == fights_df["fighter_1"]).mean()
f2_win = (fights_df["winner"] == fights_df["fighter_2"]).mean()
draw_nc = (fights_df["winner"] == "Draw/NC").mean()

print(f"✅ Rebuilt {len(fights_df)} fights with correct corner assignment")
print(f"   Skipped (no detail page): {skipped}")
print(f"\n   Red corner (F1) win rate:  {f1_win:.1%}")
print(f"   Blue corner (F2) win rate: {f2_win:.1%}")
print(f"   Draw/NC rate:              {draw_nc:.1%}")

fights_df.to_csv(f"{DATA_DIR}/fights.csv", index=False)
print(f"\n✅ Saved corrected fights.csv")
fights_df.head(10)

In [ ]:
# Cell 8 — Final Verification
print("=" * 50)
print("SCRAPING SUMMARY")
print("=" * 50)
print(f"Events:         {len(events_df)}")
print(f"Fights (raw):   {len(fights_raw_df)}")
print(f"Fights (fixed): {len(fights_df)}")
print(f"Fighters:       {len(fighters_df)}")
print(f"Fight Details:  {len(fight_details_list)}")
f1_win = (fights_df["winner"] == fights_df["fighter_1"]).mean()
print(f"\nRed corner win rate: {f1_win:.1%} (this is our baseline)")
print(f"\nSample (F1=Red Corner, F2=Blue Corner):")
print(fights_df[["fighter_1", "fighter_2", "winner", "weight_class", "method"]].head(10).to_string())
print(f"\nFiles saved in: {DATA_DIR}/")
print(f"  - events.csv")
print(f"  - fights_raw.csv   (winner-first order from event pages)")
print(f"  - fights.csv       (red corner-first order from detail pages)")
print(f"  - fighters.csv")
print(f"  - fight_details.json")